# Public InstanSeg resolution comparison on SLIDE-0330

This notebook evaluates the public `fluorescence_nuclei_and_cells` model on identical random crops with three output treatments:

1. **Unresolved:** independent nuclear and cell heads (`resolve_cell_and_nucleus=False`).
2. **Native resolve:** the standard InstanSeg biological reconciliation.
3. **Watershed resolve:** the experimental local nucleus-seeded watershed applied to the unresolved output.

All three use the same public model, input channels, normalization, and postprocessing settings. The notebook reports object counts, hole diagnostics, and side-by-side mask overlays; it does not claim that visual cleanup improves biological accuracy without ground truth.

## 1. Imports, paths, and controls

In [ ]:
import gc
import json
import os
import re
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import ndimage as ndi
from skimage.segmentation import find_boundaries, watershed
import tifffile
import torch
import zarr

SLIDE_ID = 'SLIDE-0330'
FULL_MERGE_OME = Path(
    '/data1/lowes/ratnayn/Data/CellDive_analysis_data/image_data/'
    'SLIDE-0330/outputs_v3/SLIDE-0330_full_merge.ome.tif'
)
TRAINING_ROOT = Path(os.environ.get(
    'INSTANSEG_TRAINING_ROOT', '/data1/lowes/ratnayn/Data/instanseg'
)).expanduser().resolve()
SOURCE_ROOT = Path(os.environ.get(
    'INSTANSEG_EVAL_SOURCE_ROOT',
    '/data1/lowes/ratnayn/Data/instanseg/slurm_runs/'
    'instanseg_multihead_0325_20260825/source/instanseg',
)).expanduser().resolve()
PUBLIC_MODEL_CACHE_ROOT = Path(os.environ.get(
    'INSTANSEG_PUBLIC_MODEL_CACHE',
    str(TRAINING_ROOT / 'public_model_cache'),
)).expanduser().resolve()
PUBLIC_MODEL_NAME = 'fluorescence_nuclei_and_cells'
PUBLIC_MODEL_VERSION = '0.1.1'
RESULTS_ROOT = Path(os.environ.get(
    'INSTANSEG_PUBLIC_RESOLUTION_RESULTS',
    '/data1/lowes/ratnayn/Codex/codex-scratch/mIF-pipeline/'
    'slide0330_public_resolution_comparison',
)).expanduser().resolve()
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

DEVICE = os.environ.get('INSTANSEG_VISUALIZATION_DEVICE', 'cuda:0')
if not torch.cuda.is_available():
    raise RuntimeError('Select a CUDA notebook kernel before running model inference.')

# The same deterministic crops are used by all three resolution treatments.
CROP_SIZE_PX = 768
N_CROPS = 6
RANDOM_SEED = 3300330
SAVE_FIGURES = True

# These are the public model's embedded postprocessing defaults. They are
# shared by unresolved and native-resolved calls; only the resolve flag changes.
POSTPROCESSING = {
    'min_size': 10,
    'mask_threshold': 0.53,
    'peak_distance': 5,
    'seed_threshold': 0.7,
    'overlap_threshold': 0.3,
    'mean_threshold': 0.0,
    'fg_threshold': 0.5,
    'window_size': 32,
    'cleanup_fragments': True,
}

METHODS = ('unresolved', 'native', 'watershed')
METHOD_LABELS = {
    'unresolved': 'Unresolved dual heads',
    'native': 'Native InstanSeg resolve',
    'watershed': 'Custom watershed resolve',
}
print({'slide': FULL_MERGE_OME, 'device': DEVICE, 'methods': METHOD_LABELS})

## 2. Load the public model through the native API

In [ ]:
if not FULL_MERGE_OME.is_file():
    raise FileNotFoundError(FULL_MERGE_OME)
if not (SOURCE_ROOT / 'instanseg').is_dir():
    raise FileNotFoundError(SOURCE_ROOT / 'instanseg')
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

import instanseg
from instanseg import InstanSeg as NativeInstanSeg
from tiffslide import TiffSlide
import instanseg.inference_class as instanseg_inference_class

# Preserve the validated reader substitution used by the watershed tests.
instanseg_inference_class.TiffSlide = TiffSlide

model_index_path = SOURCE_ROOT / 'instanseg' / 'bioimageio_models' / 'model-index.json'
entries = [
    entry for entry in json.loads(model_index_path.read_text())
    if entry.get('name') == PUBLIC_MODEL_NAME
]
if not entries or entries[0].get('version') != PUBLIC_MODEL_VERSION:
    raise RuntimeError(
        f'Expected {PUBLIC_MODEL_NAME} version {PUBLIC_MODEL_VERSION}.'
    )
PUBLIC_MODEL_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['INSTANSEG_BIOIMAGEIO_PATH'] = str(PUBLIC_MODEL_CACHE_ROOT)

runner = NativeInstanSeg(
    model_type=PUBLIC_MODEL_NAME, device=DEVICE, verbosity=0, channels_last=False
)
network = runner.instanseg.eval()
MODEL_PIXEL_SIZE_UM = float(network.pixel_size)
if abs(MODEL_PIXEL_SIZE_UM - 0.5) > 1e-6:
    raise ValueError(f'Expected public model scale 0.5, got {MODEL_PIXEL_SIZE_UM}.')

model_defaults = {
    name.removeprefix('default_'): getattr(network, name)
    for name in [
        'default_min_size', 'default_mask_threshold', 'default_peak_distance',
        'default_seed_threshold', 'default_overlap_threshold',
        'default_mean_threshold', 'default_fg_threshold',
        'default_window_size', 'default_cleanup_fragments',
    ]
}
display(pd.DataFrame([
    {'setting': key, 'notebook_value': POSTPROCESSING[key], 'model_default': value}
    for key, value in model_defaults.items() if key in POSTPROCESSING
]))
print({
    'instanseg_source': str(Path(instanseg.__file__).resolve()),
    'model_pixel_size_um': MODEL_PIXEL_SIZE_UM,
    'cells_and_nuclei': bool(network.cells_and_nuclei),
})

## 3. Read the common channels and choose identical crops

In [ ]:
SEGMENTATION_CHANNELS = [
    'R1_DAPI', 'R4_P19_POLYRAT', 'R4_GFP_POLY_AF488',
    'R6_CD45_CST_AF647', 'R6_PANCK_AE1_AE3_750',
    'R12_CD31_D8V9E_AF750', 'R7_NAK_ATPASE_555',
    'R8_PODOPLANIN_750', 'R8_F480_D2S9R_555',
    'R9_CD68_E3O7V_488', 'R12_CD3E_E4T1B_AF555',
]
SOURCE_PIXEL_SIZE_UM = 0.325

def channel_names_from_ome_xml(xml):
    return re.findall(r'<Channel\b[^>]*Name="([^"]+)"', xml or '')

with tifffile.TiffFile(FULL_MERGE_OME) as tf:
    source_names = channel_names_from_ome_xml(tf.ome_metadata)
    level0 = tf.series[0].levels[0]
    source_shape = tuple(int(v) for v in level0.shape)
    source_channel_to_index = {name: i for i, name in enumerate(source_names)}

missing = [name for name in SEGMENTATION_CHANNELS if name not in source_channel_to_index]
if missing:
    raise ValueError(f'Missing SLIDE-0330 channels: {missing}')
CHANNEL_IDS = [source_channel_to_index[name] for name in SEGMENTATION_CHANNELS]
DAPI_INDEX = SEGMENTATION_CHANNELS.index('R1_DAPI')
DISPLAY_INDEX = {
    'red': SEGMENTATION_CHANNELS.index('R6_CD45_CST_AF647'),
    'green': SEGMENTATION_CHANNELS.index('R7_NAK_ATPASE_555'),
    'blue': SEGMENTATION_CHANNELS.index('R1_DAPI'),
}
print({'source_shape': source_shape, 'selected_channel_ids': CHANNEL_IDS})

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
source_height, source_width = source_shape[-2:]
if CROP_SIZE_PX >= source_height or CROP_SIZE_PX >= source_width:
    raise ValueError(f'Crop size {CROP_SIZE_PX} does not fit {source_shape[-2:]}')

candidate_count = max(30, N_CROPS * 10)
candidate_coords = []
candidate_scores = []
with tifffile.TiffFile(FULL_MERGE_OME) as tf:
    store = tf.series[0].aszarr(level=0)
    try:
        source = zarr.open(store, mode='r')
        for _ in range(candidate_count):
            x = int(rng.integers(0, source_width - CROP_SIZE_PX))
            y = int(rng.integers(0, source_height - CROP_SIZE_PX))
            dapi = np.asarray(source[CHANNEL_IDS[DAPI_INDEX], y:y + CROP_SIZE_PX, x:x + CROP_SIZE_PX])
            candidate_coords.append((x, y))
            candidate_scores.append(float(np.percentile(dapi, 99.5)))
        scores = np.asarray(candidate_scores)
        eligible = np.flatnonzero(scores >= np.quantile(scores, 0.5))
        chosen = rng.choice(eligible, size=min(N_CROPS, len(eligible)), replace=False)
        if len(chosen) < N_CROPS:
            raise RuntimeError('Not enough eligible crops were generated.')
        CROP_SPECS = []
        CROP_IMAGES = []
        for crop_number, candidate_index in enumerate(chosen, start=1):
            x, y = candidate_coords[int(candidate_index)]
            crop = np.stack([
                np.asarray(source[channel_id, y:y + CROP_SIZE_PX, x:x + CROP_SIZE_PX])
                for channel_id in CHANNEL_IDS
            ]).astype(np.float32, copy=False)
            CROP_SPECS.append({
                'crop': crop_number, 'x': x, 'y': y,
                'width': CROP_SIZE_PX, 'height': CROP_SIZE_PX,
                'dapi_99_5': float(candidate_scores[int(candidate_index)]),
            })
            CROP_IMAGES.append(np.ascontiguousarray(crop))
    finally:
        store.close()
display(pd.DataFrame(CROP_SPECS))
print('Every method will receive these exact crop coordinates and channel order.')

## 4. Custom watershed and hole-detection helpers

The watershed branch is applied only to the unresolved output. A qualifying nucleus covers strictly more than half of its own pixels inside a cell. Only cells with multiple qualifying nuclei are split; qualifying nucleus pixels are then restored into the final cell mask. A hole is counted only when it is an enclosed zero-valued region inside one cell instance. The `nucleus_pixels` column identifies the requested case where that hole overlaps a predicted nucleus.

In [ ]:
def associate_nuclei_to_cells(cell_labels, nucleus_labels, min_overlap_fraction=0.5):
    both = (cell_labels > 0) & (nucleus_labels > 0)
    columns = ['cell_id', 'nucleus_id', 'overlap_pixels', 'nucleus_area', 'overlap_fraction']
    if not np.any(both):
        return pd.DataFrame(columns=columns)
    cell_ids = cell_labels[both].astype(np.int64, copy=False)
    nucleus_ids = nucleus_labels[both].astype(np.int64, copy=False)
    pair_shape = (int(cell_labels.max()) + 1, int(nucleus_labels.max()) + 1)
    pair_codes = np.ravel_multi_index((cell_ids, nucleus_ids), pair_shape)
    unique_codes, overlap_pixels = np.unique(pair_codes, return_counts=True)
    pair_cell_ids, pair_nucleus_ids = np.unravel_index(unique_codes, pair_shape)
    positive_nuclei = nucleus_labels[nucleus_labels > 0]
    ids, areas = np.unique(positive_nuclei, return_counts=True)
    area_by_id = pd.Series(areas, index=ids)
    table = pd.DataFrame({
        'cell_id': pair_cell_ids.astype(np.uint32),
        'nucleus_id': pair_nucleus_ids.astype(np.uint32),
        'overlap_pixels': overlap_pixels,
    })
    table['nucleus_area'] = table['nucleus_id'].map(area_by_id).astype(np.int64)
    table['overlap_fraction'] = table['overlap_pixels'] / table['nucleus_area']
    return table.loc[table['overlap_fraction'] > min_overlap_fraction, columns].copy()

def combine_slices(slices, margin, shape):
    y0 = max(0, min(s[0].start for s in slices) - margin)
    y1 = min(shape[0], max(s[0].stop for s in slices) + margin)
    x0 = max(0, min(s[1].start for s in slices) - margin)
    x1 = min(shape[1], max(s[1].stop for s in slices) + margin)
    return slice(y0, y1), slice(x0, x1)

def apply_local_watershed(raw_cells, raw_nuclei, qualifying_pairs):
    raw_cell_ids = np.unique(raw_cells[raw_cells > 0]).astype(np.uint32)
    qualifying_by_cell = {
        int(cell_id): group['nucleus_id'].astype(int).tolist()
        for cell_id, group in qualifying_pairs.groupby('cell_id', sort=True)
    }
    cell_slices = ndi.find_objects(raw_cells)
    nucleus_slices = ndi.find_objects(raw_nuclei)
    output = raw_cells.copy()
    next_cell_id = int(raw_cells.max()) + 1
    nucleus_to_final_cell = {}
    records = []

    for cell_id in raw_cell_ids.astype(int):
        associated = qualifying_by_cell.get(cell_id, [])
        if len(associated) == 1:
            nucleus_to_final_cell[associated[0]] = cell_id
        elif len(associated) > 1:
            object_slices = [cell_slices[cell_id - 1]] + [
                nucleus_slices[nucleus_id - 1] for nucleus_id in associated
            ]
            if not all(item is not None for item in object_slices):
                continue
            bbox = combine_slices(object_slices, margin=8, shape=raw_cells.shape)
            cell_crop, nucleus_crop = raw_cells[bbox], raw_nuclei[bbox]
            territory = cell_crop == cell_id
            markers = np.zeros(territory.shape, dtype=np.int32)
            for marker_id, nucleus_id in enumerate(associated, start=1):
                nucleus_mask = nucleus_crop == nucleus_id
                territory |= nucleus_mask
                markers[nucleus_mask] = marker_id
            split = watershed(
                -ndi.distance_transform_edt(territory),
                markers=markers,
                mask=territory,
            )
            daughter_ids = [cell_id] + list(
                range(next_cell_id, next_cell_id + len(associated) - 1)
            )
            next_cell_id += len(associated) - 1
            local_output = output[bbox]
            parent_pixels = cell_crop == cell_id
            local_output[parent_pixels] = 0
            for marker_id, (nucleus_id, daughter_id) in enumerate(
                zip(associated, daughter_ids), start=1
            ):
                local_output[(split == marker_id) & parent_pixels] = daughter_id
                nucleus_to_final_cell[nucleus_id] = daughter_id
            output[bbox] = local_output
            records.append({
                'parent_cell_id': cell_id,
                'n_nuclei': len(associated),
                'n_daughters': len(daughter_ids),
            })

    for nucleus_id, final_cell_id in nucleus_to_final_cell.items():
        output[raw_nuclei == nucleus_id] = final_cell_id
    return output, pd.DataFrame(records)

def count_instances(labels):
    return int(np.unique(labels[labels > 0]).size)

def hole_table(cell_labels, nucleus_labels, crop_number, method):
    rows = []
    for cell_id, bbox in enumerate(ndi.find_objects(cell_labels), start=1):
        if bbox is None:
            continue
        local_cells = cell_labels[bbox]
        instance = local_cells == cell_id
        holes = ndi.binary_fill_holes(instance) & (~instance) & (local_cells == 0)
        labeled_holes, n_holes = ndi.label(holes)
        local_nuclei = nucleus_labels[bbox]
        for hole_id, hole_bbox in enumerate(ndi.find_objects(labeled_holes), start=1):
            if hole_bbox is None:
                continue
            hole = labeled_holes[hole_bbox] == hole_id
            if not np.any(hole):
                continue
            hole_nuclei = local_nuclei[hole_bbox][hole]
            nucleus_ids, nucleus_counts = np.unique(hole_nuclei[hole_nuclei > 0], return_counts=True)
            yy, xx = np.nonzero(hole)
            rows.append({
                'crop': int(crop_number), 'method': method,
                'cell_id': int(cell_id), 'hole_id': int(hole_id),
                'hole_pixels': int(hole.sum()),
                'nucleus_pixels': int((hole_nuclei > 0).sum()),
                'nucleus_ids': [int(value) for value in nucleus_ids],
                'nucleus_pixel_counts': [int(value) for value in nucleus_counts],
                'center_y': int((yy.min() + yy.max()) / 2 + bbox[0].start + hole_bbox[0].start),
                'center_x': int((xx.min() + xx.max()) / 2 + bbox[1].start + hole_bbox[1].start),
            })
    columns = [
        'crop', 'method', 'cell_id', 'hole_id', 'hole_pixels',
        'nucleus_pixels', 'nucleus_ids', 'nucleus_pixel_counts',
        'center_y', 'center_x',
    ]
    return pd.DataFrame(rows, columns=columns).sort_values(
        ['nucleus_pixels', 'hole_pixels'], ascending=False
    ).reset_index(drop=True)

def hole_mask_for_record(cell_labels, row):
    cell_id = int(row['cell_id'])
    bbox = ndi.find_objects(cell_labels)[cell_id - 1]
    local = cell_labels[bbox]
    instance = local == cell_id
    holes = ndi.binary_fill_holes(instance) & (~instance) & (local == 0)
    labeled_holes, _ = ndi.label(holes)
    output = np.zeros(cell_labels.shape, dtype=bool)
    output[bbox] = labeled_holes == int(row['hole_id'])
    return output

## 5. Run all three treatments on every identical crop

In [ ]:
def predict_public(crop, resolve_cell_and_nucleus):
    kwargs = dict(POSTPROCESSING)
    kwargs['resolve_cell_and_nucleus'] = resolve_cell_and_nucleus
    with torch.inference_mode():
        labels = runner.eval_small_image(
            torch.from_numpy(crop),
            pixel_size=SOURCE_PIXEL_SIZE_UM,
            normalise=True,
            return_image_tensor=False,
            target='all_outputs',
            rescale_output=True,
            **kwargs,
        )
    labels = labels.squeeze(0).to(torch.int32).cpu().numpy()
    if labels.shape != (2, *crop.shape[-2:]):
        raise ValueError(f'Unexpected output {labels.shape}; crop={crop.shape}')
    return labels

PREDICTIONS = {}
HOLE_TABLES = {}
WATERSHED_RECORDS = {}
started = time.perf_counter()
for crop_number, crop in enumerate(CROP_IMAGES, start=1):
    print(f'Crop {crop_number}/{len(CROP_IMAGES)}', flush=True)
    unresolved = predict_public(crop, resolve_cell_and_nucleus=False)
    native = predict_public(crop, resolve_cell_and_nucleus=True)
    raw_nuclei, raw_cells = unresolved[0], unresolved[1]
    qualifying_pairs = associate_nuclei_to_cells(raw_cells, raw_nuclei)
    watershed_cells, watershed_records = apply_local_watershed(
        raw_cells, raw_nuclei, qualifying_pairs
    )
    PREDICTIONS[crop_number] = {
        'unresolved': {'nuclei': raw_nuclei, 'cells': raw_cells},
        'native': {'nuclei': native[0], 'cells': native[1]},
        'watershed': {'nuclei': raw_nuclei, 'cells': watershed_cells},
    }
    WATERSHED_RECORDS[crop_number] = watershed_records
    for method in METHODS:
        HOLE_TABLES[(crop_number, method)] = hole_table(
            PREDICTIONS[crop_number][method]['cells'],
            PREDICTIONS[crop_number][method]['nuclei'],
            crop_number, method,
        )
    print({
        'unresolved_nuclei': count_instances(raw_nuclei),
        'unresolved_cells': count_instances(raw_cells),
        'native_cells': count_instances(native[1]),
        'watershed_cells': count_instances(watershed_cells),
        'watershed_splits': len(watershed_records),
    }, flush=True)

print(f'Inference and watershed processing completed in {(time.perf_counter() - started) / 60:.1f} minutes.')
del runner, network
gc.collect()
torch.cuda.empty_cache()

## 6. Quantify masks and nucleus-cutout holes

`hole_cells_with_nucleus` counts distinct cell IDs containing at least one enclosed zero region that overlaps a predicted nucleus. This is more specific than simply counting irregular cell boundaries or all holes.

In [ ]:
summary_rows = []
all_holes = []
for crop_number in range(1, len(CROP_IMAGES) + 1):
    for method in METHODS:
        result = PREDICTIONS[crop_number][method]
        holes = HOLE_TABLES[(crop_number, method)]
        all_holes.append(holes)
        nuclear_holes = holes.loc[holes['nucleus_pixels'] > 0]
        summary_rows.append({
            'crop': crop_number, 'method': METHOD_LABELS[method],
            'nuclei': count_instances(result['nuclei']),
            'cells': count_instances(result['cells']),
            'watershed_splits': len(WATERSHED_RECORDS[crop_number]) if method == 'watershed' else np.nan,
            'holes': len(holes),
            'hole_pixels': int(holes['hole_pixels'].sum()),
            'hole_cells_with_nucleus': int(nuclear_holes['cell_id'].nunique()),
            'nucleus_cutout_holes': len(nuclear_holes),
            'nucleus_cutout_pixels': int(nuclear_holes['nucleus_pixels'].sum()),
        })
summary = pd.DataFrame(summary_rows)
display(summary)
display(summary.groupby('method', sort=False)[
    ['nuclei', 'cells', 'holes', 'hole_pixels', 'hole_cells_with_nucleus',
     'nucleus_cutout_holes', 'nucleus_cutout_pixels']
].sum(numeric_only=True))

hole_records = pd.concat(all_holes, ignore_index=True) if all_holes else pd.DataFrame()
summary.to_csv(RESULTS_ROOT / 'resolution_summary.csv', index=False)
hole_records.to_csv(RESULTS_ROOT / 'cell_hole_records.csv', index=False)
print('Saved:', RESULTS_ROOT / 'resolution_summary.csv')
print('Saved:', RESULTS_ROOT / 'cell_hole_records.csv')

## 7. Side-by-side visual comparison

In [ ]:
def robust01(image, percentiles=(1.0, 99.8)):
    image = np.asarray(image, dtype=np.float32)
    low, high = np.percentile(image, percentiles)
    if high <= low:
        return np.zeros_like(image, dtype=np.float32)
    return np.clip((image - low) / (high - low), 0, 1)

def display_rgb(crop):
    return np.stack([
        robust01(crop[DISPLAY_INDEX['red']]),
        robust01(crop[DISPLAY_INDEX['green']]),
        robust01(crop[DISPLAY_INDEX['blue']]),
    ], axis=-1)

def show_overlay(ax, rgb, result, mode='both', title=''):
    ax.imshow(rgb, interpolation='nearest')
    if mode in ('both', 'nuclei'):
        boundary = find_boundaries(result['nuclei'], mode='outer')
        ax.contour(boundary, levels=[0.5], colors=['cyan'], linewidths=0.45)
    if mode in ('both', 'cells'):
        boundary = find_boundaries(result['cells'], mode='outer')
        ax.contour(boundary, levels=[0.5], colors=['yellow'], linewidths=0.45)
    ax.set_title(title, fontsize=9)
    ax.axis('off')

for crop_number, crop in enumerate(CROP_IMAGES, start=1):
    rgb = display_rgb(crop)
    fig, axes = plt.subplots(3, 4, figsize=(40, 30), squeeze=False)
    axes[0, 0].imshow(rgb); axes[0, 0].axis('off'); axes[0, 0].set_title('Input')
    axes[1, 0].imshow(rgb); axes[1, 0].axis('off'); axes[1, 0].set_title('Input')
    axes[2, 0].imshow(rgb); axes[2, 0].axis('off'); axes[2, 0].set_title('Input')
    axes[0, 0].set_ylabel('Nuclei + cells')
    axes[1, 0].set_ylabel('Nuclei')
    axes[2, 0].set_ylabel('Cells')
    for column, method in enumerate(METHODS, start=1):
        result = PREDICTIONS[crop_number][method]
        show_overlay(axes[0, column], rgb, result, 'both', METHOD_LABELS[method])
        show_overlay(axes[1, column], rgb, result, 'nuclei', 'Nuclei')
        show_overlay(axes[2, column], rgb, result, 'cells', 'Cells')
    fig.suptitle(
        f'{SLIDE_ID} crop {crop_number} | cyan=nuclei, yellow=cells | '
        f"x={CROP_SPECS[crop_number - 1]['x']}, y={CROP_SPECS[crop_number - 1]['y']}",
        fontsize=13,
    )
    fig.tight_layout()
    if SAVE_FIGURES:
        output_path = RESULTS_ROOT / f'{SLIDE_ID}_crop{crop_number}_resolution_comparison.png'
        fig.savefig(output_path, dpi=160, bbox_inches='tight')
        print('Saved:', output_path)
    plt.show()

## 8. Locate the hole cells

In [ ]:
for crop_number, crop in enumerate(CROP_IMAGES, start=1):
    rgb = display_rgb(crop)
    fig, axes = plt.subplots(1, 3, figsize=(21, 7), squeeze=False)
    for ax, method in zip(axes[0], METHODS):
        ax.imshow(rgb)
        holes = HOLE_TABLES[(crop_number, method)]
        if not holes.empty:
            ordinary = holes.loc[holes['nucleus_pixels'] == 0]
            cutout = holes.loc[holes['nucleus_pixels'] > 0]
            if not ordinary.empty:
                ax.scatter(ordinary['center_x'], ordinary['center_y'], s=20, facecolors='none', edgecolors='orange', linewidths=0.8)
            if not cutout.empty:
                ax.scatter(cutout['center_x'], cutout['center_y'], s=26, facecolors='none', edgecolors='red', linewidths=1.0)
        ax.set_title(
            f"{METHOD_LABELS[method]}\n"
            f"red=nucleus-overlapping holes; orange=other holes"
            f"\ncutout cells={holes.loc[holes['nucleus_pixels'] > 0, 'cell_id'].nunique()}",
            fontsize=10,
        )
        ax.axis('off')
    fig.suptitle(f'{SLIDE_ID} crop {crop_number}: hole locations', fontsize=13)
    fig.tight_layout()
    if SAVE_FIGURES:
        output_path = RESULTS_ROOT / f'{SLIDE_ID}_crop{crop_number}_hole_locations.png'
        fig.savefig(output_path, dpi=160, bbox_inches='tight')
        print('Saved:', output_path)
    plt.show()

## 9. Inspect local examples of nucleus-cutout cells

Change `EXAMPLE_CROP` to inspect another crop. The table printed first is the exact list of enclosed holes, sorted so holes overlapping nuclei appear first.

In [ ]:
EXAMPLE_CROP = 1
MAX_EXAMPLES_PER_METHOD = 3
EXAMPLE_HALF_WIDTH = 96

def centered_bbox(center_y, center_x, half_width, shape):
    return (
        slice(max(0, int(center_y) - half_width), min(shape[0], int(center_y) + half_width + 1)),
        slice(max(0, int(center_x) - half_width), min(shape[1], int(center_x) + half_width + 1)),
    )

for method in METHODS:
    examples = HOLE_TABLES[(EXAMPLE_CROP, method)]
    examples = examples.loc[examples['nucleus_pixels'] > 0].head(MAX_EXAMPLES_PER_METHOD)
    display(examples)
    crop = CROP_IMAGES[EXAMPLE_CROP - 1]
    rgb = display_rgb(crop)
    result = PREDICTIONS[EXAMPLE_CROP][method]
    for _, row in examples.iterrows():
        hole_mask = hole_mask_for_record(result['cells'], row)
        view = centered_bbox(row['center_y'], row['center_x'], EXAMPLE_HALF_WIDTH, hole_mask.shape)
        local_rgb = rgb[view]
        local_hole = hole_mask[view]
        fig, axes = plt.subplots(1, 4, figsize=(20, 5))
        axes[0].imshow(local_rgb); axes[0].set_title('Morphology'); axes[0].axis('off')
        show_overlay(axes[1], local_rgb, {
            'nuclei': result['nuclei'][view], 'cells': result['cells'][view]
        }, mode='cells', title='Cell boundary')
        show_overlay(axes[2], local_rgb, {
            'nuclei': result['nuclei'][view], 'cells': result['cells'][view]
        }, mode='nuclei', title='Nucleus boundary')
        axes[3].imshow(local_rgb)
        axes[3].contour(find_boundaries(result['cells'][view], mode='outer'), levels=[0.5], colors=['yellow'], linewidths=0.7)
        axes[3].contour(find_boundaries(result['nuclei'][view], mode='outer'), levels=[0.5], colors=['cyan'], linewidths=0.7)
        hole_rgba = np.zeros((*local_hole.shape, 4), dtype=float)
        hole_rgba[local_hole] = (1, 0, 0, 0.75)
        axes[3].imshow(hole_rgba); axes[3].set_title('Red = enclosed hole'); axes[3].axis('off')
        fig.suptitle(
            f"{METHOD_LABELS[method]} | crop {EXAMPLE_CROP} | cell {int(row['cell_id'])} | "
            f"hole={int(row['hole_pixels'])} px | nucleus overlap={int(row['nucleus_pixels'])} px | "
            f"nuclei={row['nucleus_ids']}",
            fontsize=12,
        )
        fig.tight_layout()
        if SAVE_FIGURES:
            output_path = RESULTS_ROOT / (
                f'{SLIDE_ID}_crop{EXAMPLE_CROP}_{method}_'
                f'cell{int(row["cell_id"])}_hole{int(row["hole_id"])}.png'
            )
            fig.savefig(output_path, dpi=180, bbox_inches='tight')
            print('Saved:', output_path)
        plt.show()

## Interpretation

A lower hole count means the cell raster is more filled, not necessarily that the cell boundary is more correct. Native resolution can use both heads to coordinate IDs and boundaries. The custom watershed changes only ambiguous multiply-nucleated cell territories and restores qualifying nuclei; it does not add proxy cells for unmatched nuclei. Compare the local overlays with the morphology and, if needed, rerun on additional crops before choosing a downstream mask policy.

In [ ]:
provenance = {
    'slide_id': SLIDE_ID,
    'source_image': str(FULL_MERGE_OME),
    'source_pixel_size_um': SOURCE_PIXEL_SIZE_UM,
    'model': PUBLIC_MODEL_NAME,
    'model_version': PUBLIC_MODEL_VERSION,
    'model_pixel_size_um': MODEL_PIXEL_SIZE_UM,
    'channels': SEGMENTATION_CHANNELS,
    'channel_ids': CHANNEL_IDS,
    'crop_specs': CROP_SPECS,
    'random_seed': RANDOM_SEED,
    'postprocessing': POSTPROCESSING,
    'methods': METHOD_LABELS,
    'watershed_rule': 'nucleus overlap fraction strictly greater than 0.5; local distance-transform watershed for cells with multiple qualifying nuclei',
}
(RESULTS_ROOT / 'provenance.json').write_text(json.dumps(provenance, indent=2) + '\n')
print('Saved:', RESULTS_ROOT / 'provenance.json')